# 03 — CineSense AI: Full RAG Pipeline with Gemini LLM
### Cortex-RAG Series | github.com/ather-ops/Cortex-RAG
---

## What This Notebook Does

This is the **complete production RAG pipeline** for CineSense AI.

Every step from raw Netflix data to a working AI that answers in natural language.

```
Netflix_Dataset.csv
        ↓
Missing value cleaning
        ↓
Sentence-level chunking (NLTK)
        ↓
Embeddings (SentenceTransformers)
        ↓
ChromaDB PersistentClient (stored to disk)
        ↓
Semantic search with metadata filters
        ↓
Gemini 2.5 Flash LLM response
        ↓
CineSense AI answers your query
```

**What is RAG?**
- **R** = Retrieval — ChromaDB finds real Netflix titles matching your query
- **A** = Augmented — retrieved titles are injected into the LLM prompt as context
- **G** = Generation — Gemini LLM generates a natural language answer using that real context

Without RAG: LLM hallucinates fake titles with high confidence.
With RAG: LLM answers only from your real Netflix database.

**Prerequisites:**
```bash
pip install pandas numpy sentence-transformers chromadb google-generativeai nltk tiktoken seaborn matplotlib
```

**API Key:** Get a free Gemini key at [aistudio.google.com](https://aistudio.google.com)


---
## Step 1 — Import Libraries

We import everything needed for the full pipeline.
Each library has one specific job.


In [ ]:
# ─── Core data libraries ───────────────────────────────────────────
import pandas as pd          # DataFrame operations — loading, cleaning, iterating
import numpy as np           # Numerical operations — array math, embeddings manipulation

# ─── Visualization ────────────────────────────────────────────────
import seaborn as sns         # Statistical plots — bar charts, count plots
from matplotlib import pyplot as plt  # Core plotting — figures, subplots

# ─── NLP and embeddings ────────────────────────────────────────────
import nltk                  # Natural Language Toolkit — sentence tokenization
from nltk.tokenize import sent_tokenize  # Splits text into individual sentences
from sentence_transformers import SentenceTransformer  # Pretrained embedding model

# ─── Vector database ──────────────────────────────────────────────
import chromadb              # ChromaDB — stores and queries embedding vectors

# ─── LLM (Gemini) ─────────────────────────────────────────────────
import google.generativeai as genai  # Google Gemini API

# ─── Utilities ────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')  # Suppress deprecation warnings for clean output

# ─── Download NLTK sentence tokenizer data ─────────────────────────
# punkt and punkt_tab are required for sent_tokenize() to work
# These are small language model files for sentence boundary detection
nltk.download('punkt',    quiet=True)
nltk.download('punkt_tab',quiet=True)

print("All libraries imported successfully.")
print("NLTK punkt tokenizer ready.")


---
## Step 2 — Load Data with Error Handling

We load `Netflix_Dataset.csv` — the real Netflix titles dataset.

**Why error handling?**
Production code never assumes the file exists.
- `FileNotFoundError` — file not found at the path
- `Exception` — any other unexpected error (corrupted file, encoding issue)

If either error occurs, we print a clear message and exit cleanly.


In [ ]:
try:
    df = pd.read_csv("Netflix_Dataset.csv")
    print(f"Dataset loaded successfully")
    print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
    print(f"Columns: {df.columns.tolist()}")
    print()
    print(df.head(5))

except FileNotFoundError:
    print("Error: Netflix_Dataset.csv not found.")
    print("Make sure the file is in the same folder as this notebook.")
    raise

except Exception as e:
    print(f"Unexpected error loading data: {e}")
    raise


---
## Step 3 — Basic EDA (Exploratory Data Analysis)

Before touching any data, we understand it.

**Why EDA first?**
- Discover which columns have missing values
- Understand data types — what needs cleaning
- See the distribution of content types, years, ratings
- Find duplicates that could pollute our embeddings

This step does not change the data — it only observes.


In [ ]:
print("=" * 60)
print("BASIC EDA")
print("=" * 60)

print("\nDataset info:")
df.info()

print("\nDescriptive statistics:")
print(df.describe())

print("\nMissing values per column:")
print(df.isnull().sum())

print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"Unique content types: {df['type'].unique()}")
print(f"Year range: {df['release_year'].min()} to {df['release_year'].max()}")


---
## Step 4 — Fill Missing Values

We clean missing values using a smart type-aware strategy.

**The logic:**
| Column Type | Method | Why |
|-------------|--------|-----|
| Numeric — year columns | Median | Avoids decimal years, robust to outliers |
| Numeric — other | Mean | Standard imputation for continuous values |
| Text / Object | `"unknown"` | Preserves categorical integrity |

**Why not just drop rows?**
Netflix has ~8800 titles. Dropping rows with ANY missing value could lose hundreds of valid titles.
Filling is almost always better than dropping for production pipelines.

**The `analysis()` function** loops through every column automatically.
No hardcoding column names — it works on any dataset.


In [ ]:
def analysis(df: pd.DataFrame) -> pd.DataFrame:
    """
    Fill missing values across all columns using type-aware strategy.

    - Year columns (name contains 'year'): fill with median (int — no decimals)
    - Other numeric columns: fill with mean
    - Text / object columns: fill with 'unknown'

    Args:
        df: Input DataFrame with possible missing values

    Returns:
        DataFrame with all missing values filled
    """
    for col in df.columns:
        missing = df[col].isnull().sum()
        if missing == 0:
            continue  # skip columns that are already clean

        if df[col].dtype in ['int64', 'float64']:
            if 'year' in col.lower():
                # Median for year columns — no float years like 2015.7
                fill_val = int(df[col].median())
                method = "median"
            else:
                fill_val = round(df[col].mean(), 4)
                method = "mean"
        else:
            fill_val = "unknown"
            method = "constant"

        df[col] = df[col].fillna(fill_val)
        print(f"  Filled {missing:>4} missing in '{col}' using {method} = {fill_val}")

    return df

# Apply cleaning
df = analysis(df.copy())

print()
print("Missing values after cleaning:")
print(df.isnull().sum())
print()
print("Dataset is clean and ready for embeddings.")


---
## Step 5 — EDA Visualizations

Now we visualize the cleaned data.

**Why visualize after cleaning?**
- We want to see the real distribution, not the one distorted by missing values
- Charts help identify patterns that affect our RAG search quality

**5 charts:**
1. **Pie chart** — Movies vs TV Shows split
2. **Bar chart** — Top 10 content-producing countries
3. **Bar chart** — Top 10 most popular genres
4. **Line chart** — Content release growth over years
5. **Count plot** — Audience rating distribution


In [ ]:
sns.set_theme(style="whitegrid")
PRIMARY   = "#2ECC71"
SECONDARY = "#06b6d4"
ACCENT    = "#f97316"
DARK      = "#0f172a"

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.patch.set_facecolor('#03040f')
fig.suptitle('Netflix Content — CineSense AI Data Dashboard',
             color='white', fontsize=16, fontweight='bold', y=1.01)

for ax in axes.flat:
    ax.set_facecolor('#0f172a')
    ax.tick_params(colors='#94a3b8')
    for spine in ax.spines.values():
        spine.set_color('#1e293b')

# Chart 1 — Movies vs TV Shows
type_counts = df['type'].value_counts()
axes[0,0].pie(type_counts.values,
              labels=type_counts.index,
              autopct='%1.1f%%',
              colors=[PRIMARY, SECONDARY],
              textprops={'color':'white'})
axes[0,0].set_title('Movies vs TV Shows', color='white', fontweight='bold')

# Chart 2 — Top 10 countries
top_countries = df[df['country'] != 'unknown']['country'].value_counts().head(10)
axes[0,1].barh(top_countries.index[::-1], top_countries.values[::-1],
               color=PRIMARY, alpha=0.85)
axes[0,1].set_title('Top 10 Content-Producing Countries', color='white', fontweight='bold')
axes[0,1].set_xlabel('Number of Titles', color='#94a3b8')
axes[0,1].tick_params(colors='#94a3b8')
for sp in axes[0,1].spines.values(): sp.set_color('#1e293b')

# Chart 3 — Top 10 genres
genre_counts = df['listed_in'].str.split(',').explode().str.strip().value_counts().head(10)
axes[0,2].barh(genre_counts.index[::-1], genre_counts.values[::-1],
               color=SECONDARY, alpha=0.85)
axes[0,2].set_title('Top 10 Most Popular Genres', color='white', fontweight='bold')
axes[0,2].set_xlabel('Count', color='#94a3b8')
axes[0,2].tick_params(colors='#94a3b8')
for sp in axes[0,2].spines.values(): sp.set_color('#1e293b')

# Chart 4 — Release year trend
yearly = df.groupby('release_year')['show_id'].count().reset_index()
axes[1,0].plot(yearly['release_year'], yearly['show_id'],
               color=PRIMARY, linewidth=2.5, marker='o', markersize=3)
axes[1,0].fill_between(yearly['release_year'], yearly['show_id'],
                        color=PRIMARY, alpha=0.15)
axes[1,0].set_title('Content Release Growth Over Years', color='white', fontweight='bold')
axes[1,0].set_xlabel('Year', color='#94a3b8')
axes[1,0].set_ylabel('Titles', color='#94a3b8')
axes[1,0].tick_params(colors='#94a3b8')
axes[1,0].grid(True, alpha=0.15)
for sp in axes[1,0].spines.values(): sp.set_color('#1e293b')

# Chart 5 — Rating distribution
rating_counts = df['rating'].value_counts()
axes[1,1].bar(rating_counts.index, rating_counts.values,
              color=[ACCENT if i==0 else SECONDARY if i==1 else '#64748b'
                     for i in range(len(rating_counts))], alpha=0.85)
axes[1,1].set_title('Audience Rating Distribution', color='white', fontweight='bold')
axes[1,1].set_xlabel('Rating', color='#94a3b8')
axes[1,1].tick_params(colors='#94a3b8', axis='x', rotation=45)
for sp in axes[1,1].spines.values(): sp.set_color('#1e293b')

# Chart 6 — Movies vs Shows over time
movies = df[df['type']=='Movie'].groupby('release_year').size()
shows  = df[df['type']=='TV Show'].groupby('release_year').size()
axes[1,2].plot(movies.index, movies.values, color=PRIMARY, label='Movies', linewidth=2)
axes[1,2].plot(shows.index,  shows.values,  color=SECONDARY, label='TV Shows', linewidth=2)
axes[1,2].set_title('Movies vs TV Shows Over Time', color='white', fontweight='bold')
axes[1,2].set_xlabel('Year', color='#94a3b8')
axes[1,2].tick_params(colors='#94a3b8')
axes[1,2].legend(facecolor='#0f172a', labelcolor='white')
axes[1,2].grid(True, alpha=0.15)
for sp in axes[1,2].spines.values(): sp.set_color('#1e293b')

plt.tight_layout()
plt.savefig('cinesense_eda_dashboard.png', dpi=120, bbox_inches='tight', facecolor='#03040f')
plt.show()
print("EDA Dashboard saved as cinesense_eda_dashboard.png")


---
## Step 6 — Sentence-Level Chunking

This is the **core upgrade** that makes CineSense AI production-grade.

**Why chunk?**

Without chunking:
- Each Netflix title = one large combined text block
- Embeddings represent the whole block — important details get diluted
- A query about "animation style" might miss a title where that detail is buried at the end

With chunking (2 sentences per chunk):
- Each title is split into 2-sentence segments
- Each chunk gets its own precise embedding
- Queries match the most relevant chunk, not the whole noisy block
- Search results are more accurate and targeted

**How `sent_tokenize()` works:**
NLTK's sentence tokenizer uses language models (punkt) to detect sentence boundaries.
It handles abbreviations, numbers, and punctuation correctly — much better than simple `.split('.')`.

**Chunk ID pattern:** `s1_chunk_0`, `s1_chunk_1` etc.
This links every chunk back to its original Netflix title via `show_id`.


In [ ]:
def sentence_chunk(text: str, max_sentences: int = 2) -> list:
    """
    Split text into chunks of max_sentences sentences each.

    Uses NLTK sent_tokenize for accurate sentence boundary detection.
    Handles abbreviations, numbers, and punctuation correctly.

    Args:
        text: Combined text string for one Netflix title
        max_sentences: Number of sentences per chunk (default 2)

    Returns:
        List of text chunks — each is a string of max_sentences sentences
    """
    sentences = sent_tokenize(text)  # split into individual sentences
    chunks = []
    for i in range(0, len(sentences), max_sentences):
        # Join max_sentences sentences into one chunk
        chunk = " ".join(sentences[i : i + max_sentences])
        chunks.append(chunk)
    return chunks


# Build all chunks and their metadata
all_chunks: list      = []
metadata_chunks: list = []

for idx, row in df.iterrows():
    # Combine all relevant text fields into one string
    # Each field contributes to the semantic meaning of the chunk
    combined_parts = [
        str(row["title"])       if pd.notnull(row["title"])       else "",
        # title: most important — explicit name matching
        str(row["director"])    if pd.notnull(row["director"])    else "",
        # director: helps queries like "movies by Nolan"
        str(row["cast"])        if pd.notnull(row["cast"])        else "",
        # cast: helps queries like "movies with Tom Hanks"
        str(row["listed_in"])   if pd.notnull(row["listed_in"])   else "",
        # listed_in: genre tags — helps mood-based queries
        str(row["description"]) if pd.notnull(row["description"]) else "",
        # description: most semantic content — what the story is about
    ]
    combined = " ".join(combined_parts).strip()

    # Split into 2-sentence chunks
    chunks = sentence_chunk(combined, max_sentences=2)

    for chunk_idx, chunk in enumerate(chunks):
        all_chunks.append(chunk)

        # Store metadata for each chunk
        # chunk_index links back to its position in the title
        # total_chunks tells us how many chunks this title produced
        metadata_chunks.append({
            "show_id":      str(row["show_id"]),
            "title":        str(row["title"]),
            "type":         str(row["type"]),
            "country":      str(row["country"]),
            "release_year": int(row["release_year"]),
            "rating":       str(row["rating"]),
            "listed_in":    str(row["listed_in"]),
            "chunk_index":  chunk_idx,
            "total_chunks": len(chunks),
        })

# Summary
avg_words = sum(len(c.split()) for c in all_chunks) / len(all_chunks)
print(f"Original titles   : {len(df):,}")
print(f"Total chunks      : {len(all_chunks):,}")
print(f"Avg chunks/title  : {len(all_chunks)/len(df):.1f}")
print(f"Avg words/chunk   : {avg_words:.1f}")
print()
print("Example chunk 0:")
print(all_chunks[0])
print()
print("Example metadata 0:")
print(metadata_chunks[0])


---
## Step 7 — Generate Embeddings

Convert every chunk into a 384-dimensional vector using `all-MiniLM-L6-v2`.

**What is an embedding?**
A vector (list of 384 numbers) that encodes the semantic meaning of text.
- "emotional drama" and "heartbreaking story" will have very similar vectors
- "comedy" and "drama" will have very different vectors
- ChromaDB uses these vectors to find semantically similar content

**Why `all-MiniLM-L6-v2`?**
- Lightweight — fast on CPU, no GPU required
- High quality — trained on massive text corpora
- 384 dimensions — small enough to be efficient, large enough to be accurate
- Free — no API key needed, runs locally

**`show_progress_bar=True`** — shows a progress bar since encoding all chunks takes time.


In [ ]:
print("Loading SentenceTransformer model: all-MiniLM-L6-v2")
print("This model runs locally — no API key needed")
print()

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Generating embeddings for all chunks...")
print("This may take a few minutes for large datasets.")
print()

embeddings = embed_model.encode(
    all_chunks,
    show_progress_bar=True,   # show progress bar
    batch_size=64             # process 64 chunks at a time for memory efficiency
)

print()
print(f"Embeddings shape : {embeddings.shape}")
print(f"  {embeddings.shape[0]:,} chunks")
print(f"  {embeddings.shape[1]} dimensions each")
print(f"Total numbers stored: {embeddings.shape[0] * embeddings.shape[1]:,}")
print()
print("Each chunk is now a point in 384-dimensional space.")
print("Semantically similar chunks are close together in this space.")


---
## Step 8 — Initialize ChromaDB (Persistent)

We use `PersistentClient` — this **saves to disk**, not just in memory.

**Why PersistentClient?**
| `chromadb.Client()` | `chromadb.PersistentClient(path=...)` |
|---------------------|--------------------------------------|
| In memory only | Saved to disk at the given path |
| Lost when Python restarts | Reloads instantly on restart |
| Must re-embed every time | No re-embedding needed |
| 8800 titles = wait every time | 8800 titles = loads in 2 seconds |

**In production:** you embed once, store permanently, query forever.

The `./chroma_data` folder is created automatically if it doesn't exist.


In [ ]:
# PersistentClient saves to disk — embeddings survive Python restarts
# path='./chroma_data' — creates this folder in your working directory
client = chromadb.PersistentClient(path='./chroma_data')

print("ChromaDB PersistentClient initialized")
print("Data will be saved to: ./chroma_data/")
print("You will NOT need to re-embed on next run")
print()

# Delete existing collection if it exists (clean start)
# This is useful when you want to rebuild from scratch
try:
    client.delete_collection(name="netflix_titles")
    print("Deleted existing collection (clean rebuild)")
except Exception:
    print("No existing collection found — creating fresh")

# Create new collection
# metadata is stored alongside the collection for documentation
collection = client.create_collection(
    name="netflix_titles",
    metadata={
        "description": "Netflix movies and TV shows with sentence-level chunks",
        "embedding_model": "all-MiniLM-L6-v2",
        "chunk_size": "2 sentences",
        "total_titles": str(len(df)),
    }
)

print()
print(f"Collection created: '{collection.name}'")
print(f"Initial count: {collection.count()}")


---
## Step 9 — Batch Insert into ChromaDB

We insert all chunks into ChromaDB in batches of 100.

**Why batches, not all at once?**
- RAM efficiency — inserting 50,000 chunks at once could crash your laptop
- Error recovery — if something fails, only the current batch is lost
- Memory management — Python garbage collects between batches

**The ID pattern:** `s1_chunk_0`, `s1_chunk_1`
- `s1` = Netflix show_id (unique per title)
- `chunk_0` = first chunk of that title
- This makes IDs globally unique across all chunks

**What ChromaDB stores per chunk:**
- `id` — unique identifier
- `embedding` — the 384-dimensional vector
- `metadata` — show_id, title, type, year, rating, genre, chunk_index
- `document` — the original text of the chunk


In [ ]:
# Generate unique IDs — one per chunk
# Pattern: {show_id}_chunk_{chunk_index}
# Example: "s1_chunk_0", "s1_chunk_1", "s2_chunk_0" etc.
ids = [f"{meta['show_id']}_chunk_{meta['chunk_index']}" for meta in metadata_chunks]

print(f"Total items to insert: {len(ids):,}")
print(f"Example IDs: {ids[:3]}")
print()

# Batch insert
BATCH_SIZE = 100  # insert 100 chunks at a time

for i in range(0, len(embeddings), BATCH_SIZE):
    end = min(i + BATCH_SIZE, len(embeddings))  # min() prevents going past the end

    collection.add(
        ids        = ids[i:end],                        # unique IDs for this batch
        embeddings = embeddings[i:end].tolist(),        # numpy -> Python list
        metadatas  = metadata_chunks[i:end],            # dict metadata per chunk
        documents  = all_chunks[i:end],                 # original text per chunk
    )

    # Print progress every 5000 chunks
    if i % 5000 == 0:
        print(f"  Inserted {i:>6,} / {len(embeddings):,} chunks...")

print()
print(f"Insertion complete.")
print(f"Total chunks in collection: {collection.count():,}")
print()
print("ChromaDB saved to ./chroma_data/")
print("Next time you run this notebook, you can skip Steps 6-9")
print("and load directly with: chromadb.PersistentClient(path='./chroma_data')")


---
## Step 10 — Configure Gemini LLM

We connect Google Gemini 2.5 Flash as our LLM for response generation.

**Why Gemini 2.5 Flash?**
- Fast response time — important for a search assistant
- Free tier available — generous rate limits
- High quality reasoning — better than older flash models
- No local GPU required — runs on Google's servers

**The RAG role of Gemini here:**
ChromaDB handles the R (Retrieval).
Gemini handles the G (Generation).
The prompt template handles the A (Augmentation).

**API Key Safety:**
For production, use `os.getenv("GEMINI_API_KEY")` instead of hardcoding.


In [ ]:
import google.generativeai as genai
import os

# Option 1 (Development): Direct key — fine for local notebook
# Option 2 (Production): Environment variable — never hardcode in code pushed to GitHub
# genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

API_KEY = "YOUR_GEMINI_API_KEY_HERE"  # replace with your key from aistudio.google.com
genai.configure(api_key=API_KEY)

# Load Gemini 2.5 Flash
# gemini-2.5-flash: fast, free, production-quality for our use case
llm = genai.GenerativeModel("models/gemini-2.5-flash-preview-04-17")

print("Gemini 2.5 Flash loaded and ready")
print()
print("Quick test:")
test = llm.generate_content("Say 'CineSense AI is ready' and nothing else.")
print(test.text.strip())


---
## Step 11 — Context Builder Function

The `build_context()` function is the **A** in RAG — Augmentation.

It takes raw ChromaDB results and formats them into readable text
that the LLM can understand and reason over.

**Why deduplication (`seen` set)?**
Since we chunked each title into multiple chunks,
the same title may appear several times in the top-K results.
Without deduplication: user sees "Interstellar" listed 3 times.
With deduplication: each title appears only once — clean output.

**Why limit description to 120 chars?**
LLMs have token budgets. Shorter context = faster response = lower cost.
120 chars captures the key meaning without wasting tokens.


In [ ]:
def build_context(chroma_results: dict) -> str:
    """
    Format ChromaDB query results into readable context for the LLM prompt.

    Takes the raw dict from collection.query() and produces a
    clean formatted string with deduplicated titles.

    Args:
        chroma_results: Raw output from collection.query()

    Returns:
        Formatted string of Netflix titles — the A in RAG
    """
    context = ""
    seen = set()  # track titles already added — prevent duplicates

    # chroma_results["metadatas"][0] is a list of metadata dicts
    # [0] because query() returns results per query — we have one query
    for meta in chroma_results["metadatas"][0]:
        title = meta["title"]

        if title not in seen:      # deduplication check
            seen.add(title)        # mark as processed

            context += f"- {title} ({meta['release_year']})
"
            context += f"  Type  : {meta['type']}
"
            context += f"  Genre : {meta['listed_in']}

"

    return context

# Test it
print("Testing build_context() with a mock result:")
print("(actual test will happen in the search function)")
print()
print("Context format example:")
print("- Interstellar (2014)")
print("  Type  : Movie")
print("  Genre : Sci-Fi & Fantasy, Drama")
print()
print("  ... more titles ...")


---
## Step 12 — RAG Answer Generation

`rag_answer()` is the **G** in RAG — Generation.

It takes the query and ChromaDB results,
builds a structured prompt (Role + Context + Task),
and returns Gemini's natural language response.

**The 3-part prompt structure:**
1. **ROLE** — `"You are CineSense AI"` — tells Gemini to behave like a Netflix assistant
2. **CONTEXT** — real Netflix titles from ChromaDB — grounds the answer in real data
3. **TASK** — specific instructions — controls format and quality of output

**Why this structure produces better answers:**
Without role: Gemini gives generic movie advice using its training data (hallucination risk).
With role + context: Gemini reasons from your actual database titles only.


In [ ]:
def rag_answer(query: str, chroma_results: dict) -> str:
    """
    Generate a natural language recommendation using RAG.

    Pipeline:
        1. Build context from ChromaDB results (A — Augmentation)
        2. Construct structured 3-part prompt (Role + Context + Task)
        3. Send to Gemini and return the response (G — Generation)

    Args:
        query: User's natural language question
        chroma_results: Raw output from collection.query()

    Returns:
        Natural language recommendation string from Gemini
    """
    # Step A — Build context from retrieved titles
    context = build_context(chroma_results)

    if not context.strip():
        return "No matching titles found. Try a different query or remove filters."

    # Step B — Build 3-part structured prompt
    prompt = f"""You are CineSense AI — an intelligent Netflix recommendation assistant.

NETFLIX TITLES FOUND IN DATABASE:
{context}
USER ASKED: "{query}"

INSTRUCTIONS:
- Recommend the top 3 most relevant titles from the list above
- For each title: write one sentence explaining why it matches the user's request
- Be friendly and conversational
- Only recommend titles that appear in NETFLIX TITLES FOUND above
- Keep total response under 130 words"""

    # Step C — Generate response using Gemini
    response = llm.generate_content(prompt)
    return response.text

print("rag_answer() function defined.")
print()
print("It performs A + G of the RAG pipeline:")
print("  A = build_context() formats ChromaDB results")
print("  G = Gemini generates natural language answer from context")


---
## Step 13 — Advanced Search with Metadata Filters

`advanced_search()` is the **R** in RAG — Retrieval.

It combines semantic vector search with metadata filtering.

**Two types of filtering:**
1. **Semantic search** — SentenceTransformer finds titles similar in meaning to your query
2. **Metadata filter** — ChromaDB's `where` clause filters by type, year, rating, genre

**The `$and` operator:**
When multiple filters are active, ChromaDB requires `{"$and": [filter1, filter2, ...]}`.
When only one filter is active, use the filter directly — `$and` with one item causes an error.
When no filters, `where_filter = None` — search the full collection.

**Deduplication in display:**
Same title can appear multiple times (from different chunks).
`seen_titles` set ensures each title appears only once in printed results.


In [ ]:
def advanced_search(
    collection,
    embed_model,
    query_text: str,
    genre: str      = None,
    min_year: int   = None,
    max_year: int   = None,
    rating: str     = None,
    movie_type: str = None,
    top_k: int      = 5,
) -> dict:
    """
    Semantic search over ChromaDB with optional metadata filters.

    Combines SentenceTransformer embeddings with ChromaDB where clauses.
    Supports genre ($contains), year range ($gte/$lte), rating ($eq), type ($eq).
    Deduplicates display so each title appears once.

    Args:
        collection : ChromaDB collection object
        embed_model: SentenceTransformer model for query embedding
        query_text : Natural language search query
        genre      : Filter — genre substring (e.g. "Thriller")
        min_year   : Filter — minimum release year
        max_year   : Filter — maximum release year
        rating     : Filter — exact rating string (e.g. "TV-MA")
        movie_type : Filter — "Movie" or "TV Show"
        top_k      : Number of results to return (before deduplication)

    Returns:
        Raw ChromaDB results dict
    """
    # Step 1: Embed the query
    # model.encode returns numpy array — [0] gets first (and only) result
    query_emb = embed_model.encode([query_text])[0]

    # Step 2: Build metadata filter conditions
    conditions = []
    if genre:
        # $contains — genre field contains this substring
        # "Thriller" matches "Crime TV Shows, International TV Shows, TV Thrillers"
        conditions.append({"listed_in": {"$contains": genre}})
    if min_year:
        # $gte — release_year >= min_year
        conditions.append({"release_year": {"$gte": min_year}})
    if max_year:
        # $lte — release_year <= max_year
        conditions.append({"release_year": {"$lte": max_year}})
    if rating:
        # $eq — exact match on rating string
        conditions.append({"rating": {"$eq": rating}})
    if movie_type:
        # $eq — exact match on type ("Movie" or "TV Show")
        conditions.append({"type": {"$eq": movie_type}})

    # Step 3: Combine conditions
    if len(conditions) > 1:
        where_filter = {"$and": conditions}  # multiple filters — use $and
    elif len(conditions) == 1:
        where_filter = conditions[0]         # single filter — use directly
    else:
        where_filter = None                  # no filter — search everything

    # Step 4: Query ChromaDB
    results = collection.query(
        query_embeddings=[query_emb.tolist()],  # numpy -> list for ChromaDB
        where=where_filter,                      # None means no filter
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )

    # Step 5: Display deduplicated results
    print("\n" + "=" * 60)
    print(f"Query : {query_text}")
    if where_filter:
        print(f"Filter: {where_filter}")
    print("=" * 60)

    if not results["metadatas"][0]:
        print("No results found. Try adjusting filters or query.")
        return results

    seen_titles = set()
    rank = 1
    for i, meta in enumerate(results["metadatas"][0]):
        if meta["title"] in seen_titles:
            continue
        seen_titles.add(meta["title"])

        chunk_preview = results["documents"][0][i][:120].replace("\n", " ")
        print(f"\n{rank}. {meta['title']}  ({meta['release_year']})  {meta['rating']}")
        print(f"   Type  : {meta['type']}  |  Country: {meta['country']}")
        print(f"   Genre : {meta['listed_in']}")
        print(f"   Chunk : {chunk_preview}...")
        rank += 1

    return results

print("advanced_search() function defined.")


---
## Step 14 — Complete CineSense Pipeline

`cinesense()` ties everything together:
**R** (retrieve) → **A** (augment) → **G** (generate) in one function call.

This is the function you call from Streamlit, FastAPI, or any interface.


In [ ]:
def cinesense(query: str, **filters) -> tuple:
    """
    Complete CineSense AI RAG pipeline in one function call.

    Executes: Embed -> Retrieve (ChromaDB) -> Augment (context) -> Generate (Gemini)

    Args:
        query  : Natural language search query from user
        **filters: Optional keyword args passed to advanced_search()
                   Supported: genre, min_year, max_year, rating, movie_type

    Returns:
        Tuple of (chroma_results dict, llm_answer string)

    Example:
        results, answer = cinesense("something emotional", movie_type="Movie")
        results, answer = cinesense("thriller", min_year=2018, max_year=2022)
    """
    print(f"\nCineSense AI searching: {query}")
    print("-" * 55)

    # R — RETRIEVE: semantic search in ChromaDB
    results = advanced_search(
        collection  = collection,
        embed_model = embed_model,
        query_text  = query,
        top_k       = 5,
        **filters   # pass all keyword filters through
    )

    # A + G — AUGMENT + GENERATE: context + Gemini response
    answer = rag_answer(query, results)

    print("\n" + "=" * 55)
    print("CineSense AI (Gemini 2.5 Flash) says:")
    print("=" * 55)
    print(answer)
    print("=" * 55)

    return results, answer

print("cinesense() pipeline function defined.")
print()
print("Usage examples:")
print("  cinesense('something sad')")
print("  cinesense('thriller', movie_type='Movie', min_year=2019)")
print("  cinesense('comedy show', movie_type='TV Show')")


---
## Step 15 — Test the Complete RAG System

Run 5 different queries to verify every component works end to end.

**What each test verifies:**
1. Basic mood query — semantic search working
2. Type filter — metadata filtering working
3. Year range filter — range queries working
4. Genre filter — `$contains` operator working
5. Combined filters — `$and` compound queries working


In [ ]:
print("=" * 60)
print("TESTING CINESENSE AI — 5 QUERIES")
print("=" * 60)

# Test 1 — Basic mood search (no filters)
print("\n[TEST 1] Basic mood query — no filters")
cinesense("something emotional and heartbreaking")

# Test 2 — Type filter
print("\n\n[TEST 2] Type filter — TV Shows only")
cinesense("funny show for late night", movie_type="TV Show")

# Test 3 — Year range
print("\n\n[TEST 3] Year range filter")
cinesense("mind bending thriller", min_year=2018, max_year=2022)

# Test 4 — Genre filter
print("\n\n[TEST 4] Genre filter")
cinesense("crime drama", genre="Crime")

# Test 5 — Multiple filters combined
print("\n\n[TEST 5] Multiple filters — $and compound query")
cinesense("action movie", movie_type="Movie", min_year=2019, rating="TV-MA")


---
## Step 16 — Save Search Results to CSV

Export last search results for analysis or documentation.


In [ ]:
def save_results(chroma_results: dict, filename: str = "cinesense_results.csv") -> None:
    """Save ChromaDB search results to CSV file."""
    if chroma_results["metadatas"][0]:
        results_df = pd.DataFrame(chroma_results["metadatas"][0])
        results_df.to_csv(filename, index=False)
        print(f"Results saved to {filename}")
        print(f"Shape: {results_df.shape}")
        print(results_df[['title','release_year','type','listed_in']].head())
    else:
        print("No results to save.")

# Save last search results
last_results, _ = cinesense("family movie for weekend", movie_type="Movie")
save_results(last_results)


---
## Summary — What You Built

| Step | What It Does | Key Concept |
|------|-------------|-------------|
| 1 | Import libraries | Setup |
| 2 | Load Netflix_Dataset.csv | Error handling |
| 3 | Basic EDA | Understand before touching |
| 4 | Fill missing values | Type-aware imputation |
| 5 | 6 visualization charts | Data insights |
| 6 | Sentence-level chunking | 2 sentences per chunk for precision |
| 7 | Generate embeddings | 384-dim semantic vectors |
| 8 | ChromaDB PersistentClient | Saved to disk — no re-embedding |
| 9 | Batch insert | 100 chunks at a time — memory safe |
| 10 | Configure Gemini 2.5 Flash | LLM setup |
| 11 | build_context() | A in RAG — format real data for LLM |
| 12 | rag_answer() | G in RAG — Gemini generates grounded answer |
| 13 | advanced_search() | R in RAG — semantic + metadata retrieval |
| 14 | cinesense() | Full R → A → G pipeline |
| 15 | 5 test queries | End-to-end verification |
| 16 | Save to CSV | Export results |

---

## What Comes Next in Cortex-RAG

```
Notebook 04 — FastAPI Backend
  POST /search endpoint
  Pydantic request/response schemas
  Swagger docs at /docs

Notebook 05 — Streamlit UI
  Search bar + mood buttons
  Movie result cards
  LLM response display

Notebook 06 — Chrome Extension
  Inject into Netflix
  Call FastAPI from extension
  Glassmorphism overlay UI
```

---

Built by **Ather Assadullah** — Self-taught ML Engineer — Kashmir, India
GitHub: [ather-ops/Cortex-RAG](https://github.com/ather-ops/Cortex-RAG)
